In [4]:
from sysdata.sim.csv_futures_sim_data import csvFuturesSimData

data = csvFuturesSimData()


In [3]:
data.get_instrument_list()


['FTSECHINAH',
 'EU-MID',
 'EU-DIV30',
 'JP-REALESTATE',
 'SMI',
 'DJSTX-SMALL',
 'LUMBER-new',
 'EUROSTX200-LARGE',
 'CAD5',
 'PALLAD',
 'HIGHYIELD',
 'GBPEUR',
 'SP500_micro',
 'BOVESPA',
 'IRON',
 'LEANHOG',
 'GAS-LAST',
 'US-STAPLES',
 'ETHER-micro',
 'WHEAT_ICE',
 'VNKI',
 'SOYBEAN',
 'EU-DJ-UTIL',
 'FTSEINDO',
 'NZD',
 'ETHANOL',
 'GOLD',
 'MSCIASIA',
 'BB3M',
 'CAD2',
 'EU-DJ-TECH',
 'JPY_mini',
 'EU-TECH',
 'MIB',
 'SMI-MID',
 'GILT',
 'BTP',
 'EU-TRAVEL',
 'MSCIWORLD',
 'SWISSLEAD',
 'SOYBEAN_mini',
 'KRWUSD_mini',
 'STEEL',
 'HANGTECH',
 'NASDAQ_micro',
 'ROBUSTA',
 'VIX_mini',
 'CAD10',
 'AEX',
 'BBCOMM',
 'NASDAQ',
 'EU-HOUSE',
 'HOUSE-US',
 'NIKKEI400',
 'RUSSELL',
 'GASOILINE_micro',
 'EURIBOR-ICE',
 'CANOLA',
 'EU-HEALTH',
 'HEATOIL-ICE',
 'MSCIEAFA',
 'SUGAR16',
 'SILVER',
 'EUA',
 'COPPER-micro',
 'CHEESE',
 'CAD_micro',
 'EURCHF',
 'GAS-PEN',
 'MILKWET',
 'CAC',
 'CHF_micro',
 'GASOIL',
 'AUD',
 'EU-CHEM',
 'EU-DJ-OIL',
 'BUND',
 'EU-FOOD',
 'US-MATERIAL',
 'US10',
 '

In [ ]:
data.keys()
data["SP500_micro"]


index
1982-09-14 23:00:00     679.40
1982-09-15 23:00:00     679.90
1982-09-16 23:00:00     679.25
1982-09-17 23:00:00     678.35
1982-09-20 23:00:00     679.15
                        ...   
2024-03-28 17:00:00    5309.25
2024-03-28 18:00:00    5316.50
2024-03-28 19:00:00    5305.50
2024-03-28 20:00:00    5306.00
2024-03-28 23:00:00    5303.75
Name: price, Length: 35898, dtype: float64

In [ ]:
import pandas as pd
from sysquant.estimators.vol import robust_vol_calc


def calc_ewmac_forecast(price, Lfast, Lslow=None):

    price = price.resample("1B").last()
    if Lslow is None:
        Lslow = 4 * Lfast

    fast_ewma = price.ewm(span=Lfast).mean()
    slow_ewma = price.ewm(span=Lslow).mean()
    raw_ewmac = fast_ewma - slow_ewma

    vol = robust_vol_calc(price.diff())

    return raw_ewmac / vol


In [ ]:
instrument_code = "SP500_micro"
price = data.daily_prices(instrument_code)
ewmac = calc_ewmac_forecast(price, 32, 128)
ewmac.tail(5)


index
2024-03-22    8.367249
2024-03-25    8.575964
2024-03-26    8.771066
2024-03-27    8.856622
2024-03-28    9.108725
Freq: B, Name: price, dtype: float64

In [5]:
from matplotlib.pyplot import show

ewmac.plot()
show()


NameError: name 'ewmac' is not defined

In [27]:
from sysdata.sim.csv_futures_sim_data import csvFuturesSimData

data = csvFuturesSimData()

from systems.provided.rules.ewmac import ewmac_forecast_with_defaults as ewmac

from systems.forecasting import Rules

my_rules = Rules(ewmac)
my_rules.trading_rules()


{'rule0': TradingRule; function: <function ewmac_forecast_with_defaults at 0x121ead480>, data: data.daily_prices (args: {}) and other_args: }

In [28]:
type(data)


sysdata.sim.csv_futures_sim_data.csvFuturesSimData

In [29]:
my_rules = Rules(dict(ewmac=ewmac))

my_rules.trading_rules()


{'ewmac': TradingRule; function: <function ewmac_forecast_with_defaults at 0x121ead480>, data: data.daily_prices (args: {}) and other_args: }

In [30]:
from systems.basesystem import System

my_system = System([my_rules], data)
my_system


Private configuration '/Users/jasonli/Dev/FORKed repo/pysystemtrade/private/private_config.yaml' is missing or misconfigured; no problem if running in sim mode


System base_system with .config, .data, and .stages: rules

In [ ]:
my_system.rules.get_raw_forecast("SOFR", "ewmac").tail(5)


2026-05-21 14:50:11 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2026-05-21 14:50:11 DEBUG base_system Following instruments are marked as 'ignore_instruments': not included: ['EXAMPLE']
2026-05-21 14:50:11 DEBUG base_system Following instruments removed entirely from sim: ['Another_thing', 'EXAMPLE', 'bad_thing']
2026-05-21 14:50:11 DEBUG base_system {'stage': 'rules', 'instrument_code': 'SOFR'} Calculating raw forecast SOFR for ewmac


index
2024-03-22   -0.349853
2024-03-25   -0.378965
2024-03-26   -0.394422
2024-03-27   -0.381021
2024-03-28   -0.378662
Freq: B, Name: price, dtype: float64

In [ ]:
from systems.trading_rules import TradingRule

ewmac_rule = TradingRule(ewmac)
my_rules = Rules(dict(ewmac=ewmac_rule))
ewmac_rule


TradingRule; function: <function ewmac_forecast_with_defaults at 0x121ead480>, data: data.daily_prices (args: {}) and other_args: 

In [ ]:
ewmac_8 = TradingRule(
    (ewmac, [], dict(Lfast=8, Lslow=32))
)  ## as a tuple (function, data, other_args) notice the empty element in the middle
ewmac_32 = TradingRule(
    dict(function=ewmac, other_args=dict(Lfast=32, Lslow=128))
)  ## as a dict
my_rules = Rules(dict(ewmac8=ewmac_8, ewmac32=ewmac_32))
my_rules.trading_rules()["ewmac32"]


TradingRule; function: <function ewmac_forecast_with_defaults at 0x121ead480>, data: data.daily_prices (args: {}) and other_args: Lfast, Lslow

In [ ]:
my_system = System([my_rules], data)
my_system.rules.get_raw_forecast("SOFR", "ewmac32").tail(5)


Private configuration '/Users/jasonli/Dev/FORKed repo/pysystemtrade/private/private_config.yaml' is missing or misconfigured; no problem if running in sim mode
2026-05-21 15:02:23 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2026-05-21 15:02:23 DEBUG base_system Following instruments are marked as 'ignore_instruments': not included: ['EXAMPLE']
2026-05-21 15:02:23 DEBUG base_system Following instruments removed entirely from sim: ['Another_thing', 'EXAMPLE', 'bad_thing']
2026-05-21 15:02:23 DEBUG base_system {'stage': 'rules', 'instrument_code': 'SOFR'} Calculating raw forecast SOFR for ewmac32


index
2024-03-22   -0.349853
2024-03-25   -0.378965
2024-03-26   -0.394422
2024-03-27   -0.381021
2024-03-28   -0.378662
Freq: B, Name: price, dtype: float64

In [ ]:
from sysdata.config.configdata import Config

my_config = Config()
my_config


Config with elements: 

In [38]:
empty_rules = Rules()
my_config.trading_rules = dict(ewmac8=ewmac_8, ewmac32=ewmac_32)
my_system = System([empty_rules], data, my_config)

my_system.rules.get_raw_forecast("SOFR", "ewmac8")


Private configuration '/Users/jasonli/Dev/FORKed repo/pysystemtrade/private/private_config.yaml' is missing or misconfigured; no problem if running in sim mode
2026-05-21 15:12:45 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2026-05-21 15:12:45 DEBUG base_system Following instruments are marked as 'ignore_instruments': not included: ['EXAMPLE']
2026-05-21 15:12:45 DEBUG base_system Following instruments removed entirely from sim: ['Another_thing', 'EXAMPLE', 'bad_thing']
2026-05-21 15:12:45 DEBUG base_system {'stage': 'rules', 'instrument_code': 'SOFR'} Calculating raw forecast SOFR for ewmac8


index
1984-03-23         NaN
1984-03-26         NaN
1984-03-27         NaN
1984-03-28         NaN
1984-03-29         NaN
                ...   
2024-03-22   -0.792683
2024-03-25   -0.737074
2024-03-26   -0.648340
2024-03-27   -0.476329
2024-03-28   -0.381107
Freq: B, Name: price, Length: 10440, dtype: float64

In [ ]:
from systems.forecast_scale_cap import ForecastScaleCap

my_config.instruments = ["SOFR", "US10", "CORN", "SP500_micro"]
my_config.use_forecast_scale_estimates = True

fcs = ForecastScaleCap()
my_system = System([fcs, my_rules], data, my_config)

my_system.forecastScaleCap.get_forecast_scalar("SOFR", "ewmac32").tail(5)


Private configuration '/Users/jasonli/Dev/FORKed repo/pysystemtrade/private/private_config.yaml' is missing or misconfigured; no problem if running in sim mode
2026-05-21 15:15:18 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2026-05-21 15:15:18 DEBUG base_system Following instruments are marked as 'ignore_instruments': not included: ['EXAMPLE']
2026-05-21 15:15:18 DEBUG base_system Following instruments removed entirely from sim: ['Another_thing', 'EXAMPLE', 'bad_thing']
2026-05-21 15:15:18 DEBUG base_system {'stage': 'forecastScaleCap'} Getting cross sectional forecasts for scalar calculation for ewmac32 over CORN, SOFR, SP500_micro, US10
2026-05-21 15:15:18 DEBUG base_system {'stage': 'rules', 'instrument_code': 'CORN'} Calculating raw forecast CORN for ewmac32
2026-05-21 15:15:19 DEBUG base_system {'stage': 'rules', 'instrument_code': 'SOFR'} Calculating raw forecast SOFR for ewmac32
2026-05-21 15:15:19 DEBUG base_system {'stage': '

index
2024-03-22    2.975027
2024-03-25    2.975012
2024-03-26    2.975015
2024-03-27    2.975025
2024-03-28    2.975090
Freq: B, dtype: float64

In [ ]:
my_config.forecast_scalars = dict(ewmac8=5.3, ewmac32=2.65)
my_config.use_forecast_scale_estimates = False

my_system = System([fcs, my_rules], data, my_config)

my_system.forecastScaleCap.get_forecast_scalar("SOFR", "ewmac32").tail(5)


Private configuration '/Users/jasonli/Dev/FORKed repo/pysystemtrade/private/private_config.yaml' is missing or misconfigured; no problem if running in sim mode
2026-05-21 15:16:42 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2026-05-21 15:16:42 DEBUG base_system Following instruments are marked as 'ignore_instruments': not included: ['EXAMPLE']
2026-05-21 15:16:42 DEBUG base_system Following instruments removed entirely from sim: ['Another_thing', 'EXAMPLE', 'bad_thing']
2026-05-21 15:16:42 DEBUG base_system {'stage': 'rules', 'instrument_code': 'SOFR'} Calculating raw forecast SOFR for ewmac32


index
2024-03-22    2.65
2024-03-25    2.65
2024-03-26    2.65
2024-03-27    2.65
2024-03-28    2.65
Freq: B, dtype: float64

In [ ]:
my_system.forecastScaleCap.get_capped_forecast("SOFR", "ewmac32")


2026-05-21 15:17:00 DEBUG base_system {'stage': 'forecastScaleCap', 'instrument_code': 'SOFR'} Calculating capped forecast for SOFR ewmac32


index
1984-03-23         NaN
1984-03-26         NaN
1984-03-27         NaN
1984-03-28         NaN
1984-03-29         NaN
                ...   
2024-03-22   -0.927110
2024-03-25   -1.004257
2024-03-26   -1.045217
2024-03-27   -1.009705
2024-03-28   -1.003454
Freq: B, Length: 10440, dtype: float64

In [43]:
from systems.forecast_combine import ForecastCombine

combiner = ForecastCombine()
my_system = System([fcs, empty_rules, combiner], data, my_config)

my_system.combForecast.get_forecast_weights("SOFR").tail(5)


Private configuration '/Users/jasonli/Dev/FORKed repo/pysystemtrade/private/private_config.yaml' is missing or misconfigured; no problem if running in sim mode
2026-05-21 15:19:51 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2026-05-21 15:19:51 DEBUG base_system Following instruments are marked as 'ignore_instruments': not included: ['EXAMPLE']
2026-05-21 15:19:51 DEBUG base_system Following instruments removed entirely from sim: ['Another_thing', 'EXAMPLE', 'bad_thing']
2026-05-21 15:19:51 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'SOFR'} Calculating forecast weights for SOFR
2026-05-21 15:19:51 WARNING base_system {'stage': 'combForecast', 'instrument_code': 'SOFR'} WARNING: No forecast weights  - using equal weights of 0.500 over all 2 trading rules in system
2026-05-21 15:19:51 DEBUG base_system {'stage': 'forecastScaleCap', 'instrument_code': 'SOFR'} Calculating capped forecast for SOFR ewmac32
2026-05-21 15:1

,ewmac8,ewmac32
index,,
2024-03-22,0.5,0.5
2024-03-25,0.5,0.5
2024-03-26,0.5,0.5
2024-03-27,0.5,0.5
2024-03-28,0.5,0.5


In [44]:
my_system.combForecast.get_forecast_diversification_multiplier("SOFR").tail(5)


2026-05-21 15:19:56 INFO base_system {'stage': 'combForecast', 'instrument_code': 'SOFR'} Calculating forecast div multiplier for SOFR
2026-05-21 15:19:56 INFO base_system {'stage': 'combForecast', 'instrument_code': 'SOFR'} Calculating forecast correlations over CORN, SOFR, SP500_micro, US10
2026-05-21 15:19:56 DEBUG base_system {'stage': 'forecastScaleCap', 'instrument_code': 'CORN'} Calculating capped forecast for CORN ewmac32
2026-05-21 15:19:56 DEBUG base_system {'stage': 'rules', 'instrument_code': 'CORN'} Calculating raw forecast CORN for ewmac32
2026-05-21 15:19:56 DEBUG base_system {'stage': 'forecastScaleCap', 'instrument_code': 'CORN'} Calculating capped forecast for CORN ewmac8
2026-05-21 15:19:56 DEBUG base_system {'stage': 'rules', 'instrument_code': 'CORN'} Calculating raw forecast CORN for ewmac8
2026-05-21 15:19:56 DEBUG base_system {'stage': 'forecastScaleCap', 'instrument_code': 'SP500_micro'} Calculating capped forecast for SP500_micro ewmac32
2026-05-21 15:19:56 DE

index
2024-03-22    1.106181
2024-03-25    1.106183
2024-03-26    1.106185
2024-03-27    1.106187
2024-03-28    1.106189
Freq: B, dtype: float64